# 🧠 Week 14 Lab 
## Training Neural Networks in Practice
### *The Features Matter More Than the Classifier*

**Objectives:**
- Explore temporal neural data and see the impairment signal that mean rates discard
- Compare all methods on mean rates vs temporal features for two tasks
- Understand why LR beats MLP on temporal features
- Diagnose three common training failures from loss curves
- Implement data augmentation and understand when it helps vs hurts
- Compute saliency maps to interpret what the MLP learned
- Test drift robustness on temporal features

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_val_score, LeaveOneGroupOut, StratifiedKFold, train_test_split
)
from sklearn.metrics import accuracy_score

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
skf = StratifiedKFold(5, shuffle=True, random_state=42)
logo = LeaveOneGroupOut()

## Upload Data

Upload `week14_data.pkl` — the reaching dataset with a new key: `neural_timeseries`.

This file contains everything from Weeks 8–13, plus:
- `neural_timeseries`: (480, 4000) — 80 neurons × 50 time bins (10 ms each, 500 ms total)
- Temporal profile: baseline → ramp → peak → decay
- Impaired subjects: ~40 ms delayed onset and slower decay

All other keys (`neural_rates`, `X_raw`, `targets`, `labels`, `subjects`) are unchanged.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week14_data.pkl

In [ ]:
# Load data
with open('week14_data.pkl', 'rb') as f:
    D = pickle.load(f)

X_neural = D['neural_rates']       # (480, 80) — mean rates (Weeks 5–13)
X_temporal = D['neural_timeseries'] # (480, 4000) — 80 neurons × 50 bins (NEW)
X_emg = D['X_raw']                 # (480, 6)
y_dir = D['targets']               # 8 directions
y_bin = (D['labels'] == 'impaired').astype(int)
subjects = D['subjects']

n_neurons, n_bins = 80, 50
t_ms = np.arange(n_bins) * 10  # time axis in ms

print(f'Mean rates:  {X_neural.shape}')
print(f'Temporal:    {X_temporal.shape} = {n_neurons} neurons × {n_bins} bins')
print(f'Directions:  {len(np.unique(y_dir))}')
print(f'Healthy: {np.sum(y_bin==0)}, Impaired: {np.sum(y_bin==1)}')

> **A note on evaluation:** See the comment before Exercise 2.1 regarding our cross-validation strategy for this lab.

---
## Part 1: The Temporal Dataset 🟢

Before running any classifiers, visualise what the temporal data contains — and what mean rates throw away.

### Exercise 1.1: Temporal profiles — healthy vs impaired

Pick a neuron with good tuning for 90° reaches. Plot the trial-averaged temporal profile for healthy and impaired subjects. Mark the peak times.

**Expected result:** Healthy subjects peak earlier (~130 ms), impaired peak later (~170 ms). A ~40 ms delay that is invisible in the mean rate.

In [ ]:
# Exercise 1.1: Temporal profiles
neuron_idx = 56  # well-tuned for 90°
h_mask = (y_bin == 0) & (y_dir == 2)
i_mask = (y_bin == 1) & (y_dir == 2)

# TODO: Extract this neuron's temporal profiles for healthy and impaired
#   hint: X_temporal[mask][:, neuron_idx*n_bins:(neuron_idx+1)*n_bins]
# TODO: Compute trial averages and find peak times (np.argmax * 10 for ms)
# TODO: Plot 3 panels: (A) single trial, (B) trial average with peak markers,
#   (C) mean rate bar chart showing the timing difference is invisible
# YOUR CODE HERE

---
## Part 2: Mean Rates vs Temporal Features 🟢

Compare all methods on both feature sets for both tasks.

> **Cross-validation note:** Throughout the lecture notes, we use leave-one-subject-out (LOSO) to stay consistent with Weeks 5–13. In this lab, we switch to **stratified 5-fold CV**. The reason is purely practical: with 4,000 temporal features, training 20 LOSO folds for every method and comparison would make the lab painfully slow. Five-fold CV produces the same qualitative patterns — the rankings and relative gains are preserved — though the absolute percentages may differ by a few points. When you see a number in this notebook that doesn't exactly match the lecture, this is almost certainly why.

### Exercise 2.1: Two tasks, two stories

Evaluate NB, LR, LDA, RF, and MLP on:
1. Direction decoding (8-class) — mean rates vs temporal
2. Impairment detection (binary) — mean rates vs temporal

**Expected result:** Direction decoding is unchanged (~95% for all). Impairment detection is transformed: LR jumps ~26 points, MLP jumps ~17.

In [ ]:
# Exercise 2.1: Two tasks, two stories
methods = {
    'NB': GaussianNB(),
    'LR': LogisticRegression(C=10, max_iter=2000),
    'LDA': LinearDiscriminantAnalysis(),
    'RF': RandomForestClassifier(200, random_state=42),
    'MLP': MLPClassifier((64,32), max_iter=500, random_state=42),
}

# TODO: For each method, compute 5-fold CV accuracy on:
#   - Direction decoding: mean rates and temporal
#   - Impairment detection: mean rates and temporal
# TODO: Plot as 1×2 grouped bar chart
# YOUR CODE HERE

---
## Part 3: Features Matter More Than the Classifier 🟡

LR on temporal features beats MLP on mean rates. Why?

### Exercise 3.1: LR temporal coefficients

Fit LR on the temporal features and visualise which time bins get the highest absolute coefficients.

**Expected result:** Coefficients peak in the onset window (~40–120 ms) where healthy and impaired profiles diverge (see Exercise 1.1, Panel B).

In [ ]:
# Exercise 3.1: LR temporal coefficients
sc_temp = StandardScaler().fit_transform(X_temporal)

# TODO: Fit LR on sc_temp, y_bin
# TODO: Reshape lr.coef_[0] to (80, 50)
# TODO: Average across neurons and plot vs time
# TODO: Highlight the onset window (40–120ms)
# YOUR CODE HERE

---
## Part 3b: Practical Training — Training vs Validation 🟡

Section 3 of the lecture argues that you need to *see* the training process to control it. sklearn's `MLPClassifier` hides most of this, but we can still track training and validation accuracy epoch-by-epoch using `warm_start=True`. This reproduces **Figure 5** from the lecture.

### Exercise 3.2: Training vs validation curves

Train an MLP on temporal impairment-detection features and record both training and validation accuracy at each epoch. Identify the epoch where validation peaks — training beyond that point is pure memorisation.

**Expected result:** Validation accuracy peaks around epoch 15–25, then plateaus while training continues to climb toward 100%. This is the overfitting gap from Week 13, now amplified by 4,000 features.

In [ ]:
# Exercise 3.2: Training vs validation curves (Figure 5)
sc_temp = StandardScaler().fit_transform(X_temporal)
X_tr, X_te, y_tr, y_te = train_test_split(sc_temp, y_bin, test_size=0.3,
                                            random_state=42, stratify=y_bin)

# Train epoch-by-epoch to track overfitting
epochs = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300]
train_acc, val_acc = [], []

# TODO: Loop over epochs. For each n_ep:
#   1. Create MLPClassifier((64,32), max_iter=n_ep, random_state=42, learning_rate_init=0.0004)
#   2. Fit on X_tr, y_tr
#   3. Record training accuracy (on X_tr) and validation accuracy (on X_te)
# YOUR CODE HERE


# TODO: Find the best validation epoch
# TODO: Plot both curves (log x-scale), mark the best epoch with a vertical dashed line
# YOUR CODE HERE


---
## Part 4: Diagnosing Training Failures 🟡

The most practical skill in this lab: learning to read loss curves. Diagnose three common failures on our temporal data.

### Exercise 4.1: Learning rate too high

Train an MLP with learning_rate_init=0.1 and plot the loss curve.

**Expected result:** The loss zigzags — each step overshoots the minimum.

In [ ]:
# Exercise 4.1: LR too high
sc_temp = StandardScaler().fit_transform(X_temporal)
X_tr, X_te, y_tr, y_te = train_test_split(sc_temp, y_bin, test_size=0.3,
                                            random_state=42, stratify=y_bin)

# TODO: Train MLPClassifier((64,32), learning_rate_init=0.1, max_iter=100)
# TODO: Plot mlp.loss_curve_
# TODO: Annotate the symptom and fix
# YOUR CODE HERE

### Exercise 4.2: Learning rate too low

Train with learning_rate_init=0.00001 and compare with a good rate (0.001).

**Expected result:** Loss barely decreases — steps too small to make progress.

In [ ]:
# Exercise 4.2: LR too low
# TODO: Train with learning_rate_init=0.00001 and also with 0.001
# TODO: Plot both loss curves on the same axes
# TODO: Annotate the symptom and fix
# YOUR CODE HERE

### Exercise 4.3: No regularisation on 4,000 features

Compare training vs test accuracy with and without L2 regularisation (alpha).

**Expected result:** Without regularisation, training reaches 100% but test stalls. With alpha=0.01, the gap closes. Compare with Week 13: regularisation barely mattered on 80D. On 4,000D, it is essential.

In [ ]:
# Exercise 4.3: No regularisation vs regularised
check_epochs = [2, 5, 10, 20, 50, 100, 200, 500]

# TODO: For each epoch count, train MLP with alpha=0 and alpha=0.01
# TODO: Record training and test accuracy for both
# TODO: Plot all 4 curves (train/test × regularised/not)
# YOUR CODE HERE

---
## Part 5: Data Augmentation 🟡

Can we make 480 trials behave like 4,800? Test three augmentation strategies.

### Exercise 5.1: Three augmentation strategies

Implement and test:

1. **Gaussian noise:** ratẽ_n(t) = rate_n(t) + ε, where ε ~ Normal(0, σ²), σ = 0.1
2. **Time jitter:** ratẽ_n(t) = rate_n(t + Δt), where Δt ~ Uniform(−2, +2) bins
3. **Amplitude scaling:** ratẽ_n(t) = α × rate_n(t), where α ~ Uniform(0.8, 1.2)

**Expected result:** Noise and scaling help modestly. Jitter **hurts** because the onset delay IS the impairment signal — jittering ±20 ms obscures the ~40 ms delay the classifier relies on.

In [ ]:
# Exercise 5.1: Data augmentation
# TODO: Implement augment_noise, augment_jitter, augment_scale
#   using the equations from the lecture:
#   Noise:  ratẽ_n(t) = rate_n(t) + ε, ε ~ Normal(0, 0.1²)
#   Jitter: ratẽ_n(t) = rate_n(t + Δt), Δt ~ Uniform(-2, +2) bins
#   Scale:  ratẽ_n(t) = α × rate_n(t), α ~ Uniform(0.8, 1.2)
# TODO: For each, double the dataset (original + augmented)
# TODO: Compute 5-fold CV accuracy and plot as bar chart
# YOUR CODE HERE

---
## Part 6: Interpreting What the Network Learned 🔴

Compute saliency maps: which neurons and time bins matter to the MLP?

### Exercise 6.1: Saliency heatmaps

Compute the MLP saliency via finite differences: for each input feature, perturb it slightly and measure how much the output changes. Compare with LR coefficients as a heatmap.

**Expected result:** On this simulated data, both highlight the same onset window. The value is the technique — on real data with neuron-specific impairment effects, the MLP saliency would reveal patterns LR cannot.

In [ ]:
# Exercise 6.1: Saliency heatmaps
sc_temp = StandardScaler().fit_transform(X_temporal)
X_tr, X_te, y_tr, y_te = train_test_split(sc_temp, y_bin, test_size=0.3,
                                            random_state=42, stratify=y_bin)

# TODO: Fit LR and reshape coefs to (80, 50)
# TODO: Fit MLP and compute saliency via finite differences:
#   For each feature: perturb by eps, measure change in predict_proba
#   saliency[feat] = mean(|perturbed_proba - base_proba|) / eps
# TODO: Plot both as 80×50 heatmaps side by side
# YOUR CODE HERE

---
## Part 7: Drift Robustness on Temporal Features 🟡

Continuing the drift analysis from Weeks 9–13. How do temporal features hold up under electrode drift?

### Exercise 7.1: Drift robustness

Add Gaussian noise (σ = 0, 1, 2, 5, 10) to the temporal features and compare how LR, LDA, RF, and MLP degrade on impairment detection.

**Expected result:** LR is the most drift-robust — its 4,000 individual weights provide natural redundancy. MLP is second. RF drops sharply.

In [ ]:
# Exercise 7.1: Drift robustness on temporal features
drift_levels = [0, 1, 2, 5, 10]

# TODO: For each sigma, add noise to X_temporal
# TODO: Evaluate LR, LDA, RF, MLP at each noise level (5-fold CV)
# TODO: Plot accuracy vs drift for all methods
# YOUR CODE HERE

---
## 💭 Thought Exercise

1. LR on temporal features (99.0% LOSO) beat MLP on mean rates (79.8% LOSO) by 19 points. A colleague says: "We should always use the most complex model." Based on this week's results, how would you respond?

2. Time jitter augmentation hurt accuracy because the onset delay IS the impairment signal. Describe a dataset where time jitter *would* help (i.e., where onset timing is noise, not signal).

3. In Exercise 4.3, regularisation barely mattered on 80D (Week 13) but was essential on 4,000D. Explain why in terms of the ratio of parameters to training examples.

4. The saliency maps (Exercise 6.1) looked similar for LR and MLP on our simulated data. Describe what property real cortical data would need to have for the MLP saliency to reveal something LR coefficients cannot.

5. Looking at the drift results (Exercise 7.1), LR was more robust than MLP on temporal features. In Week 13, MLP was more robust than LDA on mean rates. Explain why the "best" method for drift changes depending on the features.